In [1]:
import pandas as pd
from PIL import Image
import io
import requests
from IPython.display import Image, display, clear_output
import ipywidgets as widgets
import json
import time

In [2]:
# load key from json 
with open('keys/omdb_key.json', 'r') as f:
    keys = json.load(f)
api_key = keys["omdb_key"]

In [3]:
# store address
ombapi_url = "http://www.omdbapi.com/"

# use variablae from json, no real key in the code
params = {
    "i": "tt3890160", # lookup by imbd id
    "apikey": api_key
}

# send the request to OMDB and wait for the reply
# actual call to the API: go fetch from this 
# response: stores everything OMDb sends back in a variable
response = requests.get(ombapi_url, params=params)
# check if the request works
print(response.status_code) 

200


In [4]:
# converts texts into a python dictionary and stored in "data"
data = response.json()
print(data)

{'Title': 'Baby Driver', 'Year': '2017', 'Rated': 'R', 'Released': '28 Jun 2017', 'Runtime': '113 min', 'Genre': 'Action, Crime, Drama', 'Director': 'Edgar Wright', 'Writer': 'Edgar Wright', 'Actors': 'Ansel Elgort, Jon Bernthal, Jon Hamm', 'Plot': 'Coerced into working for a crime boss, a young getaway driver must face the music when a doomed heist threatens his life, love, and freedom.', 'Language': 'English, American Sign Language', 'Country': 'United Kingdom, United States', 'Awards': 'Nominated for 3 Oscars. 44 wins & 67 nominations total', 'Poster': 'https://m.media-amazon.com/images/M/MV5BMjM3MjQ1MzkxNl5BMl5BanBnXkFtZTgwODk1ODgyMjI@._V1_QL75_UX380_CR0,0,380,562_.jpg', 'Ratings': [{'Source': 'Internet Movie Database', 'Value': '7.5/10'}, {'Source': 'Rotten Tomatoes', 'Value': '92%'}, {'Source': 'Metacritic', 'Value': '86/100'}], 'Metascore': '86', 'imdbRating': '7.5', 'imdbVotes': '696,803', 'imdbID': 'tt3890160', 'Type': 'movie', 'DVD': 'N/A', 'BoxOffice': '$107,825,862', 'Produ

In [5]:
print(data["Poster"])
display(Image(url=data["Poster"], width=150))

https://m.media-amazon.com/images/M/MV5BMjM3MjQ1MzkxNl5BMl5BanBnXkFtZTgwODk1ODgyMjI@._V1_QL75_UX380_CR0,0,380,562_.jpg


In [13]:
def get_poster(imdb_id):
    params = {"i": imdb_id, "apikey": api_key}
    response = requests.get("http://www.omdbapi.com/", params=params)
    data = response.json()
    if data.get("Response") == "False":
        return None

In [14]:
def get_poster(imdb_id):
    params = {"i": imdb_id, "apikey": api_key}
    try:
        response = requests.get(ombapi_url, params=params, timeout=10)
        data = response.json()
    except Exception as e:
        print("error on", imdb_id, e)
        return None

    if data.get("Response") == "False":
        print("failed:", imdb_id, data.get("Error"))
        return None
    poster = data.get("Poster")
    if poster == "N/A":
        return None
    return poster

print(get_poster("tt3890160"))   

https://m.media-amazon.com/images/M/MV5BMjM3MjQ1MzkxNl5BMl5BanBnXkFtZTgwODk1ODgyMjI@._V1_QL75_UX380_CR0,0,380,562_.jpg


In [15]:
# get poster url
test_ids = df["imdbID"].head(5)

for imdb_id in test_ids:
    print(imdb_id, get_poster(imdb_id))
    time.sleep(0.2)

NameError: name 'df' is not defined

In [ ]:
# get poster url
posters = []
top200 = df.iloc[100:200]

for i, imdb_id in enumerate(top200["imdbID"]):
    posters.append(get_poster(imdb_id))
    if i % 25 == 0:
        print(f"{i} / {len(top200)} done")
    time.sleep(0.2)

print("finished")

In [ ]:
movies = pd.read_csv("data/movies_with_posters.csv")

missing = movies[movies["Poster"].isna()]
print(len(missing), "posters still needed")
for count, (i, row) in enumerate(missing.iterrows()):
    movies.loc[i, "Poster"] = get_poster(row["imdbID"])
    if count % 25 == 0:
        print(f"{count} / {len(missing)} done")
        movies.to_csv("data/movies_with_posters.csv", index=False)
    time.sleep(0.2)

movies.to_csv("data/movies_with_posters.csv", index=False)
print(movies["Poster"].notna().sum(), "/", len(movies), "posters")

In [ ]:
poster_df = pd.DataFrame({"imdbID": top100["imdbID"], "Poster": posters})

movies = df.merge(poster_df, on="imdbID", how="left")

print(movies.shape)                     # still (449, 27)
print(movies["Poster"].isna().sum(), "movies without a poster")
movies[["Title", "Poster"]].head()

In [ ]:
movies.to_csv("data/movies_with_posters.csv", index=False)

In [16]:
movies = pd.read_csv("data/movies_with_posters.csv")